In [ ]:
# 0. Import libraries
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from optuna.distributions import CategoricalDistribution, FloatDistribution, IntDistribution
import pandas as pd
import sys
sys.path.append('../src')
from functions import RepeatedNestedCV, summarize_with_ci

In [ ]:
# 1. Load and preprocess data
data = pd.read_csv('../data/breast_cancer_cleaned.csv')

# 1.1 Split into X and y
X = data.drop(columns=['diagnosis']).values
y = data['diagnosis'].values

In [ ]:
# 2. Estimators with Undersampling
estimators_u={
        'LogisticRegression': ImbPipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('undersample', RandomUnderSampler(random_state=42)),
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(penalty='elasticnet', solver='saga', max_iter=10000))
        ]),
        'GaussianNB': ImbPipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('undersample', RandomUnderSampler(random_state=42)),
            ('clf', GaussianNB())
        ]),
        'LDA': ImbPipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('undersample', RandomUnderSampler(random_state=42)),
            ('scaler', StandardScaler()),
            ('clf', LinearDiscriminantAnalysis())
        ]),
        'SVM': ImbPipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('undersample', RandomUnderSampler(random_state=42)),
            ('scaler', StandardScaler()),
            ('clf', SVC(probability=True))
        ]),
        'RandomForest': ImbPipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('undersample', RandomUnderSampler(random_state=42)),
            ('clf', RandomForestClassifier())
        ]),
        'LightGBM': ImbPipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('undersample', RandomUnderSampler(random_state=42)),
            ('clf', LGBMClassifier())
        ])}

In [ ]:
# 2.1 Estimadores with class_weight='balanced'
estimators_b={
        'LogisticRegression': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(penalty='elasticnet', solver='saga', class_weight='balanced', max_iter=10000))
        ]),
        'GaussianNB': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('clf', GaussianNB())  # no class_weight in NB
        ]),
        'LDA': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('clf', LinearDiscriminantAnalysis())  # no class_weight param
        ]),
        'SVM': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('clf', SVC(class_weight='balanced', probability=True))
        ]),
        'RandomForest': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('clf', RandomForestClassifier(class_weight='balanced'))
        ]),
        'LightGBM': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('clf', LGBMClassifier(class_weight='balanced'))
        ])}

In [ ]:
# 3. Define hyperparameter search spaces for each estimator
param_grids = {
    'LogisticRegression': {
        'clf__C': FloatDistribution(1e-2, 10, log=True),
        'clf__l1_ratio': FloatDistribution(0.0, 1.0),
    },
    'GaussianNB': {
        'clf__var_smoothing': FloatDistribution(1e-9, 1e-7, log=True),
    },
    'LDA': {
        'clf__solver': CategoricalDistribution(['svd', 'lsqr', 'eigen']),
        'clf__shrinkage': CategoricalDistribution(['auto', None]),
    },
    'SVM': {
        'clf__C': FloatDistribution(1e-2, 10, log=True),
        'clf__kernel': CategoricalDistribution(['linear', 'rbf']),
    },
    'RandomForest': {
        'clf__n_estimators': IntDistribution(100, 300),
        'clf__max_depth': IntDistribution(3, 15),
        'clf__min_samples_split': IntDistribution(2, 10),
    },
    'LightGBM': {
        'clf__n_estimators': IntDistribution(100, 300),
        'clf__max_depth': IntDistribution(3, 15),
        'clf__learning_rate': FloatDistribution(0.01, 0.2),
    }
}

In [ ]:
# 4. Run nested cross-validation with undersampling
nested_cv_u = RepeatedNestedCV(
    estimators=estimators_u,
    param_grids=param_grids,
    R=10,
    N=5,
    K=3,
    scoring='balanced_accuracy'
)

print("=== Nested CV: Undersampling ===")
results_u = nested_cv_u.run(X, y)



In [ ]:
# 5. Save undersampling results
results_u['Method'] = 'Undersampling'
results_u.to_csv("../results/nested_cv_results_undersampling.csv", index=False)

In [ ]:
# 6. Run nested cross-validation with class_weight='balanced'
nested_cv_b = RepeatedNestedCV(
    estimators=estimators_b,
    param_grids=param_grids,
    R=10,
    N=5,
    K=3,
    scoring='balanced_accuracy'
)

print("=== Nested CV: Class Weight Balanced ===")
results_b = nested_cv_b.run(X, y)




In [ ]:
# 7. Save class weight results
results_b['Method'] = 'Class Weight Balanced'
results_b.to_csv("../results/nested_cv_results_class_weight.csv", index=False)

In [ ]:
# 8. Analyze and visualize metrics undersampling
summarize_with_ci(results_u)

In [ ]:
# 9. Analyze and visualize metrics class_weight='balanced'
summarize_with_ci(results_b)